# Spillage Model — Prototype

## 📝 Change History

| Date | Summary |
|------|----------|
| 2026-05-05 | Initial prototype: data collection, CNN training, visualization |

*Last updated by agent: 2026-05-05 00:00*

## 1 · Setup

In [ ]:
import sys, importlib, pickle
from pathlib import Path

import numpy as np
import torch
import pybullet as p
import pybullet_data

# Make sibling 'shovel' importable and add this folder
HERE = Path().resolve()
SHOVEL_DIR = HERE.parent / 'shovel'
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(SHOVEL_DIR))

import grid_utils, data_utils, model_utils, viz_utils
for mod in [grid_utils, data_utils, model_utils, viz_utils]:
    importlib.reload(mod)

from shovel_controller import ShovelController

CUBE_URDF = str(HERE / 'urdf' / 'cube.urdf')
SHOVEL_URDF = str(SHOVEL_DIR / 'urdf' / 'shovel' / 'shovelFlat.urdf')

print(f"cube urdf : {CUBE_URDF}")
print(f"shovel urdf: {SHOVEL_URDF}")

## 2 · Configuration

In [ ]:
from grid_utils import GridConfig
from data_utils import ArenaConfig, CollectionConfig

grid_cfg = GridConfig(h=32, w=32, cell_size=0.025)   # 80 cm × 80 cm crop

arena_cfg = ArenaConfig(
    x_min=0.35, x_max=1.05,
    y_min=-0.35, y_max=0.35,
    z_spawn=0.12,
    cube_size=0.02,
    spacing=0.022,
    fill_ratio=0.60,
    clusters=5,
    sigma_cells=4.0,
)

collect_cfg = CollectionConfig(
    n_arenas=3,
    paths_per_arena=4,
    steps_per_path=12,
    move_steps=160,
    settle_steps=120,
    start_pose=(0.60, 0.00, 0.00),
    x_bounds=(0.45, 0.95),
    y_bounds=(-0.25, 0.25),
    max_step_xy=0.08,
)

# n_arenas × paths_per_arena × steps_per_path = total samples
print(f"Expected samples: {collect_cfg.n_arenas * collect_cfg.paths_per_arena * collect_cfg.steps_per_path}")

## 3 · Data Collection

In [ ]:
# Connect headless; swap p.DIRECT → p.GUI for visual debugging
physics_client = data_utils.connect(gui=False)

# Load shovel, let it settle
robot_id = p.loadURDF(SHOVEL_URDF, [0.60, 0.0, 0.5], p.getQuaternionFromEuler([0, 0, 0]))
for _ in range(int(5 * 240)):       # 5 s settle
    p.stepSimulation()

controller = ShovelController(robot_id, urdf_path=SHOVEL_URDF)
print(f"ShovelController ready on robot_id={robot_id}")

In [ ]:
samples = data_utils.collect_dataset(
    controller, CUBE_URDF, grid_cfg, arena_cfg, collect_cfg, seed=7
)

print(f"Collected {len(samples)} samples")
print(f"  input shape : {samples[0]['input'].shape}")
print(f"  target shape: {samples[0]['target'].shape}")

In [ ]:
p.disconnect()

DATASET_PATH = HERE / 'output' / 'spillage_dataset.pkl'
with open(DATASET_PATH, 'wb') as f:
    pickle.dump(samples, f)
print(f"Dataset saved → {DATASET_PATH}")

## 4 · Inspect a Sample

In [ ]:
# Reload if re-running from here
if 'samples' not in dir():
    with open(DATASET_PATH, 'rb') as f:
        samples = pickle.load(f)

viz_utils.show_sample(samples[0], title=f"Arena={samples[0]['metadata']['arena_id']}  Path={samples[0]['metadata']['path_id']}  Step={samples[0]['metadata']['step_id']}")

## 5 · Train

In [ ]:
train_data, val_data = model_utils.split_samples(samples, val_ratio=0.20, seed=42)
print(f"Train: {len(train_data)}  Val: {len(val_data)}")

model, history, device = model_utils.train_model(
    train_data, val_data,
    epochs=25,
    batch_size=32,
    lr=1e-3,
)
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

In [ ]:
viz_utils.show_loss(history)

## 6 · Evaluate & Visualise

In [ ]:
import torch

model.eval()
for i in range(min(3, len(val_data))):
    s = val_data[i]
    x_t = torch.from_numpy(s['input']).float().unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x_t).squeeze(0).cpu().numpy()
    viz_utils.show_sample(
        s, pred=pred,
        title=f"[Val #{i}] Arena={s['metadata']['arena_id']}  Path={s['metadata']['path_id']}  Step={s['metadata']['step_id']}"
    )